# Full 2018 `DoubleMuon`: segment-matched DSA response and beam-halo diagnostics

Reruns all 68 2018A-D `DoubleMuon` skims through the five cosmic-veto channels with two extra histogram collections on top of the ones `runDataFull2018.ipynb` produced: `dsa_response_segmatch`, the pT(PF) vs pT(DSA) migration measured on the control muons the LJ-source cross-cleaning actually removes (segment match, not a dR match), and `dsa_tail_diag`, the beamspot displacement, DT/CSC hit split and phi x |eta| map that separate beam halo from a genuine mismeasured-pT tail. Outputs go to `RunOutputFiles/vtxResp2_2018Full/` and to EOS. The full sample list is deliberate rather than a subset: in the veto channel the mu-LJ DSA pT >= 200 GeV population is ~2800 muons over full 2018 and ~1000 of them have fewer than 25 muon-system hits, which is the sample the halo diagnostics have to divide over 16 phi x 3 |eta| bins; a 2018C + 2018D_0-9 subset holds about a fifth of that and leaves single-digit occupancy per cell. Every sample is skipped if its `.coffea` already exists, so the run is resumable and a partial output can be looked at while the rest completes. The MC counterpart is `runMCRespSegMatch2018.ipynb`.


In [1]:
print("IMPORTING MODULES")

import os, sys, subprocess, tempfile
from coffea import processor
import coffea.util
import matplotlib.pyplot as plt
import yaml

sidm_path = str(os.getcwd()).split('/sidm')[0]
if sidm_path not in sys.path:
    sys.path.insert(1, sidm_path)

from sidm.tools import utilities, scaleout, sidm_processor, llpnanoaodschema
from sidm.tools.metadata import write_run_metadata
utilities.set_plot_style()
%matplotlib inline

print("VOMSPROXY")
os.environ["X509_USER_PROXY"] = "/uscms_data/d3/murtazas/x509_proxy.pem"
scaleout.check_voms_proxy()

print("CREATE CLUSTER CLIENR")
cluster, client = scaleout.make_lpc_client(
    min_workers=10,
    max_workers=100,
    memory='4GB',
    disk='4GB',
    scheduler_options={"dashboard_address": ":8793"}
)
print('dashboard:', cluster.dashboard_link)
client.wait_for_workers(1, timeout=600)
print('first worker connected; cluster:', cluster)


print("CREATE RUNNER")
runner = processor.Runner(
    executor=processor.DaskExecutor(client=client, status=False),
    schema=llpnanoaodschema.LLPNanoAODSchema,
    skipbadfiles=True,
    chunksize=10000,
)

print("CHANNEL NAMES AND HIST COLLECTION")
# The two control-region channels, the Spread- and vtx_chi2-based cosmic vetoes, and
# both cosmic-enriched regions in one pass, so every comparison is made on identical
# events. Adding a channel costs one extra fill loop, not another pass over the data.
channel_names = [
    "cr_1egmOr1mu_lj",                            # Before
    "cr_1egmOr1mu_lj_spreadMu_spreadPfMu_CosMu",  # Spread veto + cosAlpha
    "cr_1egmOr1mu_lj_vtxChi2_CosMu",              # vertex veto + cosAlpha
    "cosmic_muons",                               # spread-inverted cosmic region
    "cosmic_muons_vtxChi2",                       # vertex-inverted cosmic region
]
run_tag = "vtxResp2_2018Full"  # full 2018 DoubleMuon, segment-matched response + halo diagnostics


print("Processor creation")
# mu_lj_vtx_chi2_dsa_pt holds histograms that were part of cosmic_veto when the outputs below
# were produced; this notebook requests both collections, so a rerun books every histogram
# its stored outputs contain (a superset for the 2018C and signal runs, which predate the
# pT x phi x nHits and pT x |eta| x nHits histograms).
hist_collections = ["cosmic_veto", "mu_lj_vtx_chi2_dsa_pt", "dsa_pf_response", "dsa_response_matrix",
                    "dsa_response_segmatch", "dsa_tail_diag"]

# The channels as they were run; selections.yaml has changed since. This argument was added
# after the outputs below were produced, so that the notebook reruns with the same selections.
selections_cfg = "configs/selections_vertexChi2_as_run.yaml"
p = sidm_processor.SidmProcessor(
    channel_names,
    hist_collections,
    # unweighted_hist=True,
    selections_cfg=selections_cfg,
)

max_files_bg = -1
max_files_data = -1
max_files_signal = -1




In [2]:
data = [
    'DoubleMuon_2018A_0',
    'DoubleMuon_2018A_1',
    'DoubleMuon_2018A_2',
    'DoubleMuon_2018B_0',
    'DoubleMuon_2018B_1',
    'DoubleMuon_2018B_2',
    'DoubleMuon_2018B_3',
    'DoubleMuon_2018B_4',
    'DoubleMuon_2018B_5',
    'DoubleMuon_2018B_6',
    'DoubleMuon_2018B_7',
    'DoubleMuon_2018B_8',
    'DoubleMuon_2018B_9',
    'DoubleMuon_2018B_10',
    'DoubleMuon_2018B_11',
    'DoubleMuon_2018B_12',
    'DoubleMuon_2018B_13',
    'DoubleMuon_2018C',
    'DoubleMuon_2018D_0',
    'DoubleMuon_2018D_1',
    'DoubleMuon_2018D_2',
    'DoubleMuon_2018D_3',
    'DoubleMuon_2018D_4',
    'DoubleMuon_2018D_5',
    'DoubleMuon_2018D_6',
    'DoubleMuon_2018D_7',
    'DoubleMuon_2018D_8',
    'DoubleMuon_2018D_9',
    'DoubleMuon_2018D_10',
    'DoubleMuon_2018D_11',
    'DoubleMuon_2018D_12',
    'DoubleMuon_2018D_13',
    'DoubleMuon_2018D_14',
    'DoubleMuon_2018D_15',
    'DoubleMuon_2018D_16',
    'DoubleMuon_2018D_17',
    'DoubleMuon_2018D_18',
    'DoubleMuon_2018D_19',
    'DoubleMuon_2018D_20',
    'DoubleMuon_2018D_21',
    'DoubleMuon_2018D_22',
    'DoubleMuon_2018D_23',
    'DoubleMuon_2018D_24',
    'DoubleMuon_2018D_25',
    'DoubleMuon_2018D_26',
    'DoubleMuon_2018D_27',
    'DoubleMuon_2018D_28',
    'DoubleMuon_2018D_29',
    'DoubleMuon_2018D_30',
    'DoubleMuon_2018D_31',
    'DoubleMuon_2018D_32',
    'DoubleMuon_2018D_33',
    'DoubleMuon_2018D_34',
    'DoubleMuon_2018D_35',
    'DoubleMuon_2018D_36',
    'DoubleMuon_2018D_37',
    'DoubleMuon_2018D_38',
    'DoubleMuon_2018D_39',
    'DoubleMuon_2018D_40',
    'DoubleMuon_2018D_41',
    'DoubleMuon_2018D_42',
    'DoubleMuon_2018D_43',
    'DoubleMuon_2018D_44',
    'DoubleMuon_2018D_45',
    'DoubleMuon_2018D_46',
    'DoubleMuon_2018D_47',
    'DoubleMuon_2018D_48',
    'DoubleMuon_2018D_49',
]
print("Processing DATA")
for x in data:
    print(x)
    dir_path = f"{sidm_path}/RunOutputFiles/{run_tag}"
    if os.path.exists(f"{dir_path}/{x}.coffea"):
        print("output exists, skipping")
        continue
    fileset = utilities.make_fileset([x],'llpNanoAOD_v2',location_cfg='data_skimmed.yaml',
                                     max_files=max_files_data,replace_xcache=True,)
    output_data = runner.run(fileset, treename='Events', processor_instance=p)
    dir_path = f"{sidm_path}/RunOutputFiles/{run_tag}"  # outside sidm/ so UploadDirectory does not ship outputs to workers
    os.makedirs(dir_path, exist_ok=True)
    coffea.util.save(output_data, f"{dir_path}/{x}.coffea"  )
    # Provenance sidecar: selections, hist collections, input file list, per-sample
    # cross section, SIDM commit and coffea version, next to the .coffea it describes.
    meta_local = write_run_metadata(
        f"{dir_path}/{x}.coffea",
        fileset=fileset,
        selections=channel_names,
        selections_cfg=selections_cfg,
        hist_collections=hist_collections,
        schema="LLPNanoAODSchema",
        chunksize=10000,
        extra={"run_tag": run_tag},
    )
    REDIR   = "root://cmseos.fnal.gov"
    EOS_DIR = f"/store/group/lpcmetx/SIDM/coffea_outputs/{os.environ['USER']}/{run_tag}"
    subprocess.run(["xrdfs", REDIR, "mkdir", "-p", EOS_DIR], check=True)
    with tempfile.TemporaryDirectory() as tmp:
        coffea_local = os.path.join(tmp, f"{x}.coffea")
        coffea.util.save(output_data, coffea_local)
        print(f"{REDIR}/{EOS_DIR}/{os.path.basename(coffea_local)}")
        subprocess.run(["xrdcp", "-f", coffea_local, f"{REDIR}/{EOS_DIR}/{os.path.basename(coffea_local)}"], 
                       check=True)
    subprocess.run(["xrdcp", "-f", meta_local,
                    f"{REDIR}/{EOS_DIR}/{os.path.basename(meta_local)}"], check=True)
    
    
   




In [3]:



bgs = []  # data-only study

for x in bgs:
    print(x)
    dir_path = f"{sidm_path}/RunOutputFiles/{run_tag}"
    if os.path.exists(f"{dir_path}/{x}.coffea"):
        print("output exists, skipping")
        continue
    fileset = utilities.make_fileset([x], "skimmed_llpNanoAOD_v2", location_cfg="backgrounds.yaml",
                                     max_files=max_files_bg,
                                     replace_xcache=True, )
    print(len(fileset[x]["files"]))
    output = runner.run(fileset, treename="Events", processor_instance=p)
    dir_path = f"{sidm_path}/RunOutputFiles/{run_tag}"  # outside sidm/ so UploadDirectory does not ship outputs to workers
    os.makedirs(dir_path, exist_ok=True)
    coffea.util.save(output, f"{dir_path}/{x}.coffea"  )
    # Provenance sidecar: selections, hist collections, input file list, per-sample
    # cross section, SIDM commit and coffea version, next to the .coffea it describes.
    meta_local = write_run_metadata(
        f"{dir_path}/{x}.coffea",
        fileset=fileset,
        selections=channel_names,
        selections_cfg=selections_cfg,
        hist_collections=hist_collections,
        schema="LLPNanoAODSchema",
        chunksize=10000,
        extra={"run_tag": run_tag},
    )
    REDIR   = "root://cmseos.fnal.gov"
    EOS_DIR = f"/store/group/lpcmetx/SIDM/coffea_outputs/{os.environ['USER']}/{run_tag}"
    subprocess.run(["xrdfs", REDIR, "mkdir", "-p", EOS_DIR], check=True)
    with tempfile.TemporaryDirectory() as tmp:
        coffea_local = os.path.join(tmp, f"{x}.coffea")
        coffea.util.save(output, coffea_local)
        print(f"{REDIR}/{EOS_DIR}/{os.path.basename(coffea_local)}")
        subprocess.run(["xrdcp", "-f", coffea_local, f"{REDIR}/{EOS_DIR}/{os.path.basename(coffea_local)}"],
                       check=True)
    subprocess.run(["xrdcp", "-f", meta_local,
                    f"{REDIR}/{EOS_DIR}/{os.path.basename(meta_local)}"], check=True)

In [4]:
client.close()

cluster.close()